# Customer Support Ticket Router with Reducer History

A small LangGraph example that routes support tickets to billing or technical handlers. A reducer appends each visited node to the graph path history. The example uses keyword rules and does not need an API key.

## Install dependencies

Run this setup cell in Google Colab before running the example.

In [ ]:
%pip install -q langgraph

## Define the state and graph

The `history` field uses `operator.add` as a reducer. Each node returns a one-item list, which LangGraph adds to the history already in the state.

In [ ]:
import operator
from typing import Annotated, Literal, TypedDict

from langgraph.graph import END, START, StateGraph


class TicketState(TypedDict):
    ticket: str
    category: str
    reply: str
    history: Annotated[list[str], operator.add]


def classify_ticket(state: TicketState) -> dict[str, object]:
    text = state["ticket"].lower()
    billing_keywords = ("bill", "charge", "payment", "refund", "invoice", "subscription")
    category = "billing" if any(word in text for word in billing_keywords) else "technical"
    return {"category": category, "history": ["classify"]}


def route_ticket(state: TicketState) -> Literal["billing", "technical"]:
    return state["category"]


def handle_billing(state: TicketState) -> dict[str, object]:
    return {
        "reply": "I can help with your billing question. Please share the invoice or charge details.",
        "history": ["billing"],
    }


def handle_technical(state: TicketState) -> dict[str, object]:
    return {
        "reply": "I can help troubleshoot this technical issue. Please share any error message and the steps to reproduce it.",
        "history": ["technical"],
    }


builder = StateGraph(TicketState)
builder.add_node("classify", classify_ticket)
builder.add_node("billing", handle_billing)
builder.add_node("technical", handle_technical)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route_ticket)
builder.add_edge("billing", END)
builder.add_edge("technical", END)

ticket_router = builder.compile()

## Draw the graph

In [ ]:
from IPython.display import Image, display

display(Image(ticket_router.get_graph().draw_mermaid_png()))

## Try one ticket on each route

In [ ]:
sample_tickets = [
    "I was charged twice for my monthly subscription.",
    "The app crashes with an error when I try to upload a file.",
]

for ticket in sample_tickets:
    result = ticket_router.invoke(
        {"ticket": ticket, "category": "", "reply": "", "history": []}
    )
    print(f"Ticket: {result['ticket']}")
    print(f"Category: {result['category']}")
    print(f"Reply: {result['reply']}")
    print(f"Visited nodes: {' -> '.join(result['history'])}\n")